# Sequential Compressibility

This notebook asks whether previous reasoning history adds predictive signal for current block-summary compression.

Unit of analysis: one block-summary pair.

Target: `high_token_compression`, created in `01_data_loading_engineering.ipynb` from the training-split block compression threshold.

Predictors: problem-derived features, metadata, current block features, block index, and history available before the current summary is observed.

Retrospective full-trace variables such as `n_blocks_in_trace` and `relative_block_position` are intentionally excluded.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.features import (
    count_share_table,
    format_metrics_percent,
    latest_feature_build_dir,
    normalize_difficulty,
)

from src.reasoning_compression.modeling import (
    GRADIENT_BOOSTING_PARAM_GRID,
    RANDOM_FOREST_PARAM_GRID,
    classification_report_frame,
    evaluate_classifier,
    fit_selected_model,
    make_preprocessor,
    random_forest_feature_importance,
    select_model_params,
)

In [2]:
FULL_BUILD_DIR = latest_feature_build_dir(Path("../data/full_feature_builds"))
TARGET = "high_token_compression"
SPLIT_COL = "model_split"
GROUP_COL = "trace_id"

df_blocks = pd.read_parquet(
    FULL_BUILD_DIR / "blocks_features_full_labeled.parquet"
)
df_blocks["difficulty"] = df_blocks["difficulty"].map(normalize_difficulty)

required_columns = {TARGET, SPLIT_COL, GROUP_COL}
missing_columns = required_columns.difference(df_blocks.columns)
if missing_columns:
    raise ValueError(
        "Block table is missing required columns: "
        f"{sorted(missing_columns)}. Re-run 01_data_loading_engineering.ipynb "
        "with RUN_FULL_BUILD = False."
    )

df_blocks.shape

(2013510, 23)

In [3]:
split_target_distribution = (
    df_blocks
    .groupby(SPLIT_COL)[TARGET]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .assign(share_pct=lambda df: (df["share"] * 100).round(2))
    .drop(columns="share")
)

split_target_distribution

,model_split,high_token_compression,share_pct
0,test,0,75.03
1,test,1,24.97
2,train,0,75.00
3,train,1,25.00


## History Features

Lagged and cumulative features are shifted so that current-row summary compression is not used to predict itself. First blocks have no previous history, so history features are filled with zero and paired with `has_previous_block`.

In [4]:
df_seq = df_blocks.sort_values([GROUP_COL, "block_index"]).copy()

df_seq["prev_summary_to_block_token_ratio"] = (
    df_seq.groupby(GROUP_COL)["summary_to_block_token_ratio"].shift(1)
)
df_seq["prev_token_compression_savings"] = (
    df_seq.groupby(GROUP_COL)["token_compression_savings"].shift(1)
)
df_seq["prev_block_tokens"] = (
    df_seq.groupby(GROUP_COL)["block_tokens"].shift(1)
)
df_seq["prev_summary_tokens"] = (
    df_seq.groupby(GROUP_COL)["summary_tokens"].shift(1)
)

df_seq["cum_block_tokens_before_t"] = (
    df_seq.groupby(GROUP_COL)["block_tokens"].cumsum()
    - df_seq["block_tokens"]
)
df_seq["cum_summary_tokens_before_t"] = (
    df_seq.groupby(GROUP_COL)["summary_tokens"].cumsum()
    - df_seq["summary_tokens"]
)
df_seq["cum_compression_ratio_before_t"] = (
    df_seq["cum_summary_tokens_before_t"]
    / df_seq["cum_block_tokens_before_t"]
).replace([np.inf, -np.inf], np.nan)

df_seq["mean_prev_summary_to_block_ratio"] = (
    df_seq
    .groupby(GROUP_COL)["summary_to_block_token_ratio"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

df_seq["has_previous_block"] = (df_seq["block_index"] > 0).astype(int)

history_features = [
    "prev_summary_to_block_token_ratio",
    "prev_token_compression_savings",
    "prev_block_tokens",
    "prev_summary_tokens",
    "cum_block_tokens_before_t",
    "cum_summary_tokens_before_t",
    "cum_compression_ratio_before_t",
    "mean_prev_summary_to_block_ratio",
]
df_seq[history_features] = df_seq[history_features].fillna(0)

df_seq[history_features + ["has_previous_block"]].head()

,prev_summary_to_block_token_ratio,prev_token_compression_savings,prev_block_tokens,prev_summary_tokens,cum_block_tokens_before_t,cum_summary_tokens_before_t,cum_compression_ratio_before_t,mean_prev_summary_to_block_ratio,has_previous_block
0,0.000000,0.000000,0.0,0.0,0,0,0.000000,0.000000,0
1,0.385000,0.615000,200.0,77.0,200,77,0.385000,0.385000,1
2,0.150611,0.849389,737.0,111.0,937,188,0.200640,0.267805,1
3,0.203364,0.796636,654.0,133.0,1591,321,0.201760,0.246325,1
4,0.269006,0.730994,342.0,92.0,1933,413,0.213658,0.251995,1


**Sequential history feature check interpretation**

The first row has no previous block, so history features are filled with zero and `has_previous_block` marks that special case. Later rows use only lagged or cumulative values from prior blocks. This is the key leakage check for the sequential setup: the current summary compression ratio is not used to predict itself.

## Feature Set

The sequential task uses only information available at block `t`: problem and metadata, the current block, current block index, and history from previous blocks. It excludes final trace length and relative position.

In [5]:
numeric_features = [
    "block_index",
    "block_chars",
    "block_tokens",
    "problem_chars",
    "problem_tokens",
    "problem_math_symbol_share",
    "problem_question_mark_count",
    "prev_summary_to_block_token_ratio",
    "prev_token_compression_savings",
    "prev_block_tokens",
    "prev_summary_tokens",
    "cum_block_tokens_before_t",
    "cum_summary_tokens_before_t",
    "cum_compression_ratio_before_t",
    "mean_prev_summary_to_block_ratio",
]

binary_features = [
    "problem_has_multiple_choice",
    "problem_has_code_fence",
    "has_previous_block",
]

categorical_features = [
    "domain",
    "source",
    "difficulty",
]

feature_columns = numeric_features + binary_features + categorical_features

retrospective_features = {"n_blocks_in_trace", "relative_block_position"}
leaked_features = retrospective_features.intersection(feature_columns)
if leaked_features:
    raise ValueError(
        "Retrospective full-trace features are not allowed here: "
        f"{sorted(leaked_features)}"
    )

In [6]:
df_model = df_seq[
    feature_columns + [TARGET, GROUP_COL, SPLIT_COL]
].dropna().copy()

X = df_model[feature_columns]
y = df_model[TARGET]
groups = df_model[GROUP_COL]
model_split = df_model[SPLIT_COL]

train_mask = model_split == "train"
test_mask = model_split == "test"

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]
groups_train = groups.loc[train_mask]
groups_test = groups.loc[test_mask]

X_train.shape, X_test.shape

((1611167, 21), (402343, 21))

In [7]:
len(set(groups_train).intersection(set(groups_test)))

0

## Modeling Utilities

Shared evaluation, grouped validation tuning, and feature-importance helpers are imported from `src.reasoning_compression.modeling`.


In [ ]:
rf_param_grid = RANDOM_FOREST_PARAM_GRID
gb_param_grid = GRADIENT_BOOSTING_PARAM_GRID

In [10]:
preprocessor = make_preprocessor(
    numeric_columns=numeric_features,
    binary_columns=binary_features,
    categorical_columns=categorical_features,
)

## Baseline

In [11]:
seq_dummy_model = DummyClassifier(strategy="most_frequent")
seq_dummy_model.fit(X_train, y_train)

seq_dummy_metrics = evaluate_classifier(seq_dummy_model, X_test, y_test)
format_metrics_percent(seq_dummy_metrics)

,metric,value
0,accuracy,75.03%
1,balanced_accuracy,50.00%
2,roc_auc,50.00%


## Random Forest

In [12]:
seq_rf_selection_results = select_model_params(
    model_name="Random forest",
    model_class=RandomForestClassifier,
    param_grid=rf_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

seq_rf_selection_results

,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",264465,211089,53376,0.751424,0.716175,0.804104
1,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",264465,211089,53376,0.751049,0.715977,0.803495
2,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",264465,211089,53376,0.767311,0.706292,0.801532
3,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",264465,211089,53376,0.766543,0.706520,0.800651


In [13]:
seq_rf_params = seq_rf_selection_results.loc[0, "params"]
seq_rf_model = fit_selected_model(
    model_class=RandomForestClassifier,
    selected_params=seq_rf_params,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
)

seq_rf_pred = seq_rf_model.predict(X_test)
seq_rf_metrics = evaluate_classifier(seq_rf_model, X_test, y_test)
format_metrics_percent(seq_rf_metrics)

,metric,value
0,accuracy,75.41%
1,balanced_accuracy,72.43%
2,roc_auc,81.14%


In [14]:
random_forest_feature_importance(seq_rf_model).head(20).round(4)

,feature,importance
0,num__block_tokens,0.2465
1,num__block_chars,0.1947
2,num__cum_compression_ratio_before_t,0.0616
3,num__mean_prev_summary_to_block_ratio,0.0555
4,num__cum_summary_tokens_before_t,0.0501
5,num__prev_summary_to_block_token_ratio,0.0452
6,num__cum_block_tokens_before_t,0.0445
7,num__problem_chars,0.0433
8,num__prev_block_tokens,0.0423
9,num__prev_summary_tokens,0.0407


**Sequential feature importance interpretation**

Current-block size remains the strongest signal, but prior-history features now enter the top ranks: cumulative compression ratio, previous mean compression, cumulative lengths, and previous block-summary ratios all contribute. This suggests that compression behavior has some within-trace persistence, although the current block still carries the largest share of predictive information.

## Gradient Boosting

In [15]:
seq_gb_selection_results = select_model_params(
    model_name="Gradient boosting",
    model_class=HistGradientBoostingClassifier,
    param_grid=gb_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

seq_gb_selection_results

,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",264465,211089,53376,0.719312,0.725488,0.806475
1,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",264465,211089,53376,0.795414,0.648431,0.806383
2,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",264465,211089,53376,0.719181,0.725859,0.806282
3,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",264465,211089,53376,0.717382,0.724893,0.806228
4,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",264465,211089,53376,0.795076,0.647927,0.806196
5,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",264465,211089,53376,0.719256,0.725960,0.806191
6,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",264465,211089,53376,0.795208,0.647352,0.806155
7,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",264465,211089,53376,0.795020,0.646643,0.806150
8,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",264465,211089,53376,0.717832,0.724887,0.806084
9,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",264465,211089,53376,0.718675,0.725141,0.806032


In [16]:
seq_gb_params = seq_gb_selection_results.loc[0, "params"]
seq_gb_model = fit_selected_model(
    model_class=HistGradientBoostingClassifier,
    selected_params=seq_gb_params,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
)

seq_gb_pred = seq_gb_model.predict(X_test)
seq_gb_metrics = evaluate_classifier(seq_gb_model, X_test, y_test)
format_metrics_percent(seq_gb_metrics)

,metric,value
0,accuracy,72.34%
1,balanced_accuracy,73.13%
2,roc_auc,81.30%


## Model Comparison

In [17]:
seq_model_results = pd.DataFrame([
    {
        "task": "Sequential compressibility",
        "feature_set": "Problem + current block + prior history + metadata",
        "model": "Dummy",
        "selected_params": None,
        **seq_dummy_metrics,
    },
    {
        "task": "Sequential compressibility",
        "feature_set": "Problem + current block + prior history + metadata",
        "model": "Random forest",
        "selected_params": seq_rf_params,
        **seq_rf_metrics,
    },
    {
        "task": "Sequential compressibility",
        "feature_set": "Problem + current block + prior history + metadata",
        "model": "Gradient boosting",
        "selected_params": seq_gb_params,
        **seq_gb_metrics,
    },
])

seq_model_results_display = seq_model_results.copy()
for metric in ["accuracy", "balanced_accuracy", "roc_auc"]:
    seq_model_results_display[metric] = (
        seq_model_results_display[metric] * 100
    ).round(2)

seq_model_results_display

,task,feature_set,model,selected_params,accuracy,balanced_accuracy,roc_auc
0,Sequential compressibility,Problem + current block + prior history + meta...,Dummy,None,75.03,50.00,50.00
1,Sequential compressibility,Problem + current block + prior history + meta...,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",75.41,72.43,81.14
2,Sequential compressibility,Problem + current block + prior history + meta...,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",72.34,73.13,81.30


**Sequential model comparison interpretation**

Sequential models produce the strongest overall results in this run, but the gain over the current-block notebook is modest rather than dramatic. Gradient boosting has the best balanced accuracy and ROC AUC here, while random forest is close. This supports the project hypothesis that reasoning history adds signal, but most predictability still comes from observing the current block.

In [18]:
seq_reports = pd.concat(
    [
        classification_report_frame("Random forest", y_test, seq_rf_pred),
        classification_report_frame("Gradient boosting", y_test, seq_gb_pred),
    ],
    ignore_index=True,
)

seq_reports.round(4)

,model,label,precision,recall,f1-score,support
0,Random forest,0,0.8754,0.7838,0.8270,301862.0
1,Random forest,1,0.5058,0.6648,0.5745,100481.0
2,Random forest,accuracy,NaN,NaN,0.7541,402343.0
3,Random forest,macro avg,0.6906,0.7243,0.7008,402343.0
4,Random forest,weighted avg,0.7831,0.7541,0.7640,402343.0
5,Gradient boosting,0,0.8947,0.7156,0.7952,301862.0
6,Gradient boosting,1,0.4665,0.7470,0.5743,100481.0
7,Gradient boosting,accuracy,NaN,NaN,0.7234,402343.0
8,Gradient boosting,macro avg,0.6806,0.7313,0.6848,402343.0
9,Gradient boosting,weighted avg,0.7878,0.7234,0.7400,402343.0


**Sequential classification report interpretation**

Gradient boosting reaches higher recall for high-compression blocks, while random forest keeps higher raw accuracy and slightly better precision for the positive class. Because the target is imbalanced, balanced accuracy and class-1 recall are more informative than accuracy alone when the practical goal is to identify compressible blocks.